In [2]:
import pandas as pd
import numpy as np

def kaufman_efficiency_ratio(close_series, period=10):
    # Variation nette sur la période (entre maintenant et il y a N périodes)
    net_change = np.abs(close_series - close_series.shift(period))
    
    # Somme des variations absolues bar à bar sur la période
    change = np.abs(close_series - close_series.shift(1))
    volatility = change.rolling(window=period).sum()
    
    # Éviter la division par zéro
    er = np.where(volatility == 0, 0, net_change / volatility)
    
    return pd.Series(er, index=close_series.index)

# Exemple d'utilisation :
# df['KER'] = kaufman_efficiency_ratio(df['Close'], period=10)

In [3]:
import numpy as np
import pandas as pd


def implement_market_structure_states_strict(df, n=5):
    """Calcule les états du marché avec transition obligatoire par l'état NEUTRAL.

    Aucun passage direct entre BULLISH et BEARISH n'est autorisé.
    """
    df = df.copy()
    length = len(df)

    df['is_swing_high'] = False
    df['is_swing_low'] = False
    for k in range(n, len(df) - n):
        # Le plus haut des n bougies précédentes et suivantes
        high_range = df['High'].iloc[k - n:k + n + 1]
        if df['High'].iloc[k] == high_range.max():
            df.loc[df.index[k], 'is_swing_high'] = True
            
        # Le plus bas des n bougies précédentes et suivantes
        low_range = df['Low'].iloc[k - n:k + n + 1]
        if df['Low'].iloc[k] == low_range.min():
            df.loc[df.index[k], 'is_swing_low'] = True

    states = []
    current_state = "NEUTRAL"

    # Variables de structure de marché
    last_confirmed_high = None
    last_confirmed_low = None
    higher_low = None  # Le plancher (uniquement actif en BULLISH)
    lower_high = None  # Le plafond (uniquement actif en BEARISH)

    for j in range(length):
        close_p = df["Close"].iloc[j]

        # --- ÉTAPE B : Machine à états avec transit strict par NEUTRAL ---
        if current_state == "BULLISH":
            # Sortie de Bullish : Clôture sous le Higher Low (HL) -> NEUTRAL uniquement
            if higher_low is not None and close_p < higher_low:
                current_state = "NEUTRAL"
                last_confirmed_high = None
                last_confirmed_low = None
                higher_low = None  # Reset des repères obsolètes
                lower_high = None

        elif current_state == "BEARISH":
            # Sortie de Bearish : Clôture au-dessus du Lower High (LH) -> NEUTRAL uniquement
            if lower_high is not None and close_p > lower_high:
                current_state = "NEUTRAL"
                last_confirmed_high = None
                last_confirmed_low = None
                higher_low = None
                lower_high = None  # Reset des repères obsolètes

        elif current_state == "NEUTRAL":
            # En zone neutre, on attend un NOUVEAU BOS pour définir la tendance
            # Entrée en Bullish : Clôture au-dessus du dernier sommet confirmé
            # il faut tjs un high and un low avant de former une structure bearish ou bullish
            if last_confirmed_high is not None and last_confirmed_low is not None and close_p > last_confirmed_high:
                current_state = "BULLISH"
                higher_low = (
                    last_confirmed_low  # On fige le dernier creux comme HL
                )

            # Entrée en Bearish : Clôture sous le dernier creux confirmé
            elif last_confirmed_high is not None and last_confirmed_low is not None and close_p < last_confirmed_low:
                current_state = "BEARISH"
                lower_high = (
                    last_confirmed_high  # On fige le dernier sommet comme LH
                )

        states.append(current_state)

         # --- ÉTAPE A : Enregistrement des Swings dès qu'ils se confirment ---
        if df["is_swing_high"].iloc[j] == True:
            last_confirmed_high = df["High"].iloc[j]
            if current_state == "BEARISH" and lower_high > last_confirmed_high:
                lower_high = last_confirmed_high

        if df["is_swing_low"].iloc[j] == True:
            last_confirmed_low = df["Low"].iloc[j]
            if current_state == "BULLISH" and higher_low < last_confirmed_low:
                higher_low = last_confirmed_low

    df["market_state"] = states
    return df


In [ ]:

from backtesting import Strategy

# global_tp_multiple = 0
global_stop_trade_hour = 0
global_start_trade_hour = 0
global_er15_threshold = 0
global_range_to_consider = 0
global_close_range_hour = 0

class OpeningRangeBreakout(Strategy):
    # variables
    range_to_consider = 3
    tp_multiple_orb = 5
    tp_multiple_rsi = 4
    start_trade_hour = 5
    stop_trade_hour = 16
    er15_threshold = 0.3
    close_range_hour = 8

    # fixed variables
    risk_percent = 0.01
    max_leverage = 4
    my_equity = 10000
    open_range_hour = close_range_hour - range_to_consider

    # def init(self, range_to_consider, tp_multiple, stop_trade_hour, close_range_hour):
    def init(self):
        self.current_day = None
        self.current_day_order_placed = False
        self.last_low = None
        self.last_high = None
        self.order_placed_hour = None
        self.order_executed = False
        self.opening_range_high = None
        self.opening_range_low = None

        if global_range_to_consider != 0:
                    self.range_to_consider = global_range_to_consider

        # if global_tp_multiple != 0:
        #     self.tp_multiple = global_tp_multiple

        if global_stop_trade_hour != 0:
            self.stop_trade_hour = global_stop_trade_hour

        if global_start_trade_hour != 0:
            self.start_trade_hour = global_start_trade_hour

        if global_er15_threshold != 0:
            self.er15_threshold = global_er15_threshold

        if global_close_range_hour != 0:
                    self.close_range_hour = global_close_range_hour
            
        if global_close_range_hour != 0 and global_range_to_consider != 0:
            self.open_range_hour = global_close_range_hour - global_range_to_consider

    def reset_range(self, day):
        # delete placed order at the end of the day if not executed
        if self.current_day_order_placed is True and self.order_placed_hour:
            for trade in self.trades:
                if trade.tag == self.order_placed_hour:
                    self.order_executed = True

            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.order_placed_hour:
                        order.cancel()

        self.current_day = day
        self.current_day_order_placed = False
        self.opening_range_high = None
        self.opening_range_low = None
        self.order_placed_hour = None
        self.last_low = None
        self.last_high = None
        self.order_executed = False

    def place_long_order(self, candle_date):
        planned_entry_price = self.data.High[-1]
        sl_price = self.last_low
        per_share_risk = abs(planned_entry_price - sl_price)
        tp_multiple = self.tp_multiple_rsi 
        tp_price = planned_entry_price + per_share_risk * tp_multiple
        position_size = int((self.risk_percent * self.equity) / per_share_risk)

        if per_share_risk < self.data['ATR'][-1] * 2.5:
            # print(f'Go long limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
            self.order_placed_hour = candle_date.time().hour
            self.buy(size = position_size, stop=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.order_placed_hour)
            self.current_day_order_placed = True
        
    def place_short_order(self, candle_date):
        planned_entry_price = self.data.Low[-1]
        sl_price = self.last_high
        per_share_risk = abs(planned_entry_price - sl_price)
        tp_multiple = self.tp_multiple_rsi 
        tp_price = planned_entry_price - per_share_risk * tp_multiple
        position_size = int((self.risk_percent * self.equity) / per_share_risk)

        if per_share_risk < self.data['ATR'][-1] * 2.5:
            # print(f'Go short limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
            self.order_placed_hour = candle_date.time().hour
            self.sell(size = position_size, stop=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.order_placed_hour)
            self.current_day_order_placed = True
            
    def use_rsi_strategy(self, candle_datetime):
        if self.last_low is None:
            self.last_low = self.data.Low[-1]
        else:
            if self.last_low > self.data.Low[-1]:
                self.last_low = self.data.Low[-1]

        if self.last_high is None:
            self.last_high = self.data.High[-1]
        else:
            if self.last_high < self.data.High[-1]:
                self.last_high = self.data.High[-1]

        c_long = self.data['RSI14'][-1] < 35
        c_short = self.data['RSI14'][-1] > 65

        c_market_structure = self.data['market_state'] == 'NEUTRAL'
        
        c_time_limit = candle_datetime.time().hour < self.stop_trade_hour and candle_datetime.time().hour > self.start_trade_hour

        if not self.current_day_order_placed:
            if c_long and c_time_limit and c_market_structure:
                self.place_long_order(candle_datetime)

            if c_short and c_time_limit and c_market_structure:
                self.place_short_order(candle_datetime)
                
        else:
            for trade in self.trades:
                if trade.tag == self.order_placed_hour:
                    self.order_executed = True

            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.order_placed_hour:
                        order.cancel()
                        self.current_day_order_placed = False

                        if c_time_limit and c_market_structure:
                            if order.is_long:
                                self.place_long_order(candle_datetime)
                            else:
                                self.place_short_order(candle_datetime)

    def use_orb_strategy(self, candle_datetime):
        if candle_datetime.time().hour < self.close_range_hour:
            return
        
        if candle_datetime.time().hour > self.close_range_hour and self.opening_range_high is not None and self.opening_range_low is not None:
            c_long = self.data.Open[-1] < self.opening_range_high and self.data.Close[-1] > self.opening_range_high and self.data['market_state'][-1] == 'BULLISH'
            c_short = self.data.Open[-1] > self.opening_range_low and self.data.Close[-1] < self.opening_range_low and self.data['market_state'][-1] == 'BEARISH'
            
            c2 = (self.opening_range_high - self.opening_range_low) < self.data['ATR'][-1] * 2.5
            c3 = (self.opening_range_high - self.opening_range_low) > self.data['ATR'][-1] * 1.0
            c4 = candle_datetime.time().hour < self.stop_trade_hour

            if not self.current_day_order_placed:
                if c_long and c2 and c3 and c4:
                    planned_entry_price = self.data.Close[-1]
                    sl_price = self.opening_range_low
                    per_share_risk = abs(planned_entry_price - sl_price)
                    tp_multiple = self.tp_multiple_orb 

                    tp_price = planned_entry_price + per_share_risk * tp_multiple
                    position_size = int((self.risk_percent * self.equity) / per_share_risk)

                    # print(f'Go long tp = {tp_price} sl = {sl_price} position size = {position_size}')
                    order = self.buy(size = position_size, tp=tp_price, sl=sl_price)
                    self.current_day_order_placed = True
                if c_short and c2 and c3 and c4:
                    planned_entry_price = self.data.Close[-1]
                    sl_price = self.opening_range_high
                    per_share_risk = abs(planned_entry_price - sl_price)
                    tp_multiple = self.tp_multiple_orb
                        
                    tp_price = planned_entry_price - per_share_risk * tp_multiple
                    position_size = int((self.risk_percent * self.equity) / per_share_risk)

                    # print(f'Go short tp = {tp_price} sl = {sl_price} position size = {position_size}')
                    order = self.sell(size = position_size, tp=tp_price, sl=sl_price)
                    self.current_day_order_placed = True

    def next(self):
        t = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()
        if (self.current_day != current_bar_date):
            self.reset_range(current_bar_date)

        if self.open_range_hour < t.time().hour and t.time().hour <= self.close_range_hour:
            if self.opening_range_high is not None:
                self.opening_range_high = max(self.opening_range_high, self.data.High[-1])
            else:
                self.opening_range_high = self.data.High[-1]

            if self.opening_range_low is not None:
                    self.opening_range_low = min(self.opening_range_low, self.data.Low[-1])
            else:
                self.opening_range_low = self.data.Low[-1]

            
        
        if self.data['ER'][-1] < self.er15_threshold:
            self.use_rsi_strategy(t)
        else :
            self.use_orb_strategy(t)

        # if self.data['ER'][-1] > self.er15_threshold:
        #     self.use_orb_strategy(t)
                            
        # print("--------------------------------")
            



In [10]:
from matplotlib import pyplot as plt
import pandas as pd
from backtesting import Backtest
from ta.volatility import AverageTrueRange
from ta.momentum import RSIIndicator

# global_tp_multiple = 3
global_stop_trade_hour = 16
global_start_trade_hour = 5
global_er15_threshold = 0.3

window = 2
global_range_to_consider = 3
global_close_range_hour = 7
global_stop_trade_hour = 18
# global_tp_multiple = 5
global_er15_threshold = 0.3
global_tp_multiple_er = 1

start_date = '2020-11-1 00:00'
end_date = '2023-2-28 00:00'

# start_date = '2020-1-1 00:00'
# end_date = '2026-9-16 00:00'
pair = 'GBPJPY'
timezone = "Europe/London"

df_d1 = pd.read_csv(f"./Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]

indicator_er = kaufman_efficiency_ratio(df_d1['Close'], 20)
df_result = implement_market_structure_states_strict(df_d1, n=window)
df_result['ER'] = indicator_er.values

df_h1 = pd.read_csv(f"./Trading Data/{pair}_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

h1_tf_direction_list = []
h1_tf_er15_list = []

for i in range(len(df_result)):
    row = df_result.iloc[i]
        
    df_by_day = df_h1.loc[f'{row.name.date()}' : f'{row.name.date()}']
    for index2, row2 in df_by_day.iterrows():
        h1_tf_direction_list.append(df_result.iloc[i-1]['market_state'])
        h1_tf_er15_list.append(df_result.iloc[i-1]['ER'])

df_h1['market_state'] = h1_tf_direction_list
df_h1['ER'] = h1_tf_er15_list

indicator_atr_14 = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr_14._atr

indicator_rsi_14 = RSIIndicator(df_h1['Close'], window=14)
df_h1['RSI14'] = indicator_rsi_14._rsi
df_h1.index = pd.to_datetime(df_h1.index, utc=True).tz_convert(timezone)

bt = Backtest(df_h1, OpeningRangeBreakout, margin=0.02, finalize_trades=True)

stats = bt.run()

print(stats)

bt.plot()

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-11-30 15:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-12-16 11:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http

Start                     2020-11-01 22:00...
End                       2023-02-28 00:00...
Duration                    848 days 02:00:00
Exposure Time [%]                     21.7673
Equity Final [$]                    17699.018
Equity Peak [$]                     18428.619
Return [%]                           76.99018
Buy & Hold Return [%]                21.38914
Return (Ann.) [%]                    21.88398
Volatility (Ann.) [%]                23.31559
CAGR [%]                             27.87511
Sharpe Ratio                           0.9386
Sortino Ratio                         2.15931
Calmar Ratio                          1.53455
Alpha [%]                            71.74842
Beta                                  0.24507
Max. Drawdown [%]                   -14.26081
Avg. Drawdown [%]                    -2.54986
Max. Drawdown Duration      177 days 05:00:00
Avg. Drawdown Duration       13 days 01:00:00
# Trades                                  174
Win Rate [%]                      

GridPlot(id='p2113', ...)

In [60]:
from matplotlib import pyplot as plt
import pandas as pd
from backtesting import Backtest
from ta.volatility import AverageTrueRange
from ta.momentum import RSIIndicator

# global_tp_multiple = 3
global_stop_trade_hour = 16
global_start_trade_hour = 5
global_er15_threshold = 0.3

window = 2
global_range_to_consider = 3
global_close_range_hour = 7
global_stop_trade_hour = 18
# global_tp_multiple = 5
global_er15_threshold = 0.3
global_tp_multiple_er = 1

# start_date = '2021-3-1 00:00'
# end_date = '2023-3-30 00:00'
start_date = '2020-1-1 00:00'
end_date = '2026-9-16 00:00'
pair = 'GBPJPY'
timezone = "Europe/London"

df_d1 = pd.read_csv(f"./Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]

indicator_er = kaufman_efficiency_ratio(df_d1['Close'], 20)
df_result = implement_market_structure_states_strict(df_d1, n=window)
df_result['ER'] = indicator_er.values

df_h1 = pd.read_csv(f"./Trading Data/{pair}_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

h1_tf_direction_list = []
h1_tf_er15_list = []

for i in range(len(df_result)):
    row = df_result.iloc[i]
    # er is previous day er
    if i == 0:
        er = row['ER']
    else :
        er = df_result.iloc[i-1]['ER']

    df_by_day = df_h1.loc[f'{row.name.date()}' : f'{row.name.date()}']
    for index2, row2 in df_by_day.iterrows():
        h1_tf_direction_list.append(row['market_state'])
        h1_tf_er15_list.append(er)

df_h1['market_state'] = h1_tf_direction_list
df_h1['ER'] = h1_tf_er15_list

indicator_atr_14 = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr_14._atr

indicator_rsi_14 = RSIIndicator(df_h1['Close'], window=14)
df_h1['RSI14'] = indicator_rsi_14._rsi
df_h1.index = pd.to_datetime(df_h1.index, utc=True).tz_convert(timezone)

bt = Backtest(df_h1, OpeningRangeBreakout, margin=0.02, finalize_trades=True)

stats = bt.run()

print(stats)

current_date = None
first_hour = None
last_hour = None
daily_equity = 0
daily_list = []
daily_equity_list = []
for index in range(len(stats._equity_curve)):
    row = stats._equity_curve.iloc[index]
    if current_date is None:
        current_date = row.name
        first_hour = row.Equity
    else:
        if current_date.date() != row.name.date():
            daily_list.append(current_date)
            daily_equity_list.append(stats._equity_curve.iloc[index-1].Equity)
            current_date = None

print(len(daily_list))         
print(len(daily_equity_list))         

with open(f"./results/COMBINE_DAILY_EQUITY_{pair}.csv", "a") as f:
    f.write(f"Date,Equity\n")

    for i in range(len(daily_list)):
        f.write(f"{daily_list[i]},{daily_equity_list[i]}\n")

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-01-27 17:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-02-19 09:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http

Start                     2020-01-01 22:00...
End                       2026-09-16 01:00...
Duration                   2449 days 02:00:00
Exposure Time [%]                    25.59535
Equity Final [$]                    56173.109
Equity Peak [$]                     56174.381
Return [%]                          461.73109
Buy & Hold Return [%]                45.36014
Return (Ann.) [%]                     23.0473
Volatility (Ann.) [%]                25.13451
CAGR [%]                             29.35486
Sharpe Ratio                          0.91696
Sortino Ratio                         2.11436
Calmar Ratio                          1.01162
Alpha [%]                           454.55735
Beta                                  0.15815
Max. Drawdown [%]                   -22.78262
Avg. Drawdown [%]                    -2.18788
Max. Drawdown Duration      405 days 14:00:00
Avg. Drawdown Duration       12 days 13:00:00
# Trades                                  500
Win Rate [%]                      